In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install twixtools
import os
filename='/kaggle/input/fiddata/meas_MID00801_FID26929_vibe963_dx_Cartesian_6echo.dat'

In [ ]:
%matplotlib inline
import twixtools
import numpy as np
import matplotlib.pyplot as plt
import os

example_dir = filename

def ifftnd(kspace, axes=[-1]):
    from numpy.fft import fftshift, ifftshift, ifftn
    if axes is None:
        axes = range(kspace.ndim)
    img = fftshift(ifftn(ifftshift(kspace, axes=axes), axes=axes), axes=axes)
    img *= np.sqrt(np.prod(np.take(img.shape, axes)))
    return img


def fftnd(img, axes=[-1]):
    from numpy.fft import fftshift, ifftshift, fftn
    if axes is None:
        axes = range(img.ndim)
    kspace = fftshift(fftn(ifftshift(img, axes=axes), axes=axes), axes=axes)
    kspace /= np.sqrt(np.prod(np.take(kspace.shape, axes)))
    return kspace

def rms_comb(sig, axis=1):
    return np.sqrt(np.sum(abs(sig)**2, axis))

In [ ]:
# parse the twix file
twix = twixtools.read_twix(filename)

# twix is a list of measurements:
print('\nnumber of separate scans (multi-raid):', len(twix))

In [ ]:
mdb=twix[1]['mdb']
print(mdb[1000])

In [ ]:
mdb=twix[1]['mdb']
print(mdb[100])

In [ ]:
print(twix[-1].keys())

In [ ]:
# sort all 'imaging' mdbs into a k-space array
image_mdbs = [mdb for mdb in twix[-1]['mdb'] if mdb.is_image_scan()]

n_line = 1 + max([mdb.cLin for mdb in image_mdbs])

# assume that all data were acquired with same number of channels & columns:
n_channel, n_column = image_mdbs[0].data.shape

kspace = np.zeros([n_line, n_channel, n_column], dtype=np.complex64)
for mdb in image_mdbs:
    kspace[mdb.cLin] = mdb.data

print('\nk-space shape', kspace.shape)

# reconstruct an image and show the result:
plt.figure(figsize=[12,8])
plt.subplot(121)
plt.title('k-space')
plt.imshow(abs(kspace[:,0])**0.2, cmap='gray', origin='lower')
plt.axis('off')

image = ifftnd(kspace, [0,-1])
image = rms_comb(image)
plt.subplot(122)
plt.title('image')
plt.imshow(abs(image), cmap='gray', origin='lower')
plt.axis('off')

In [ ]:
# Collect all values
echoes = set()
partitions = set()
lines = set()
channels = set()
columns = set()

for mdb in image_mdbs:
    echoes.add(mdb.cEco)
    partitions.add(mdb.cPar)
    lines.add(mdb.cLin)
    
    n_channel, n_column = mdb.data.shape
    channels.add(n_channel)
    columns.add(n_column)

# Print summary
print("=== Dataset summary ===")
print(f"Number of unique echoes: {len(echoes)} -> {sorted(echoes)}")
print(f"Number of unique partitions: {len(partitions)} -> {sorted(partitions)}")
print(f"Number of unique phase-encoding lines: {len(lines)} -> min={min(lines)}, max={max(lines)}")
print(f"Number of unique channels: {sorted(channels)}")
print(f"Number of unique columns: {sorted(columns)}")

# Optional: check for consistency
if len(channels) == 1 and len(columns) == 1:
    print("\nAll MDBs have the same number of channels and columns ")
else:
    print("\nSome MDBs have varying channels or columns ")

In [ ]:
mdb = image_mdbs[0]
for attr in dir(mdb):
    if not attr.startswith("__"):
        print(attr)

In [ ]:
num_mdbs = len(image_mdbs)
print(f"Total number of MDBs in dataset: {num_mdbs}")

In [ ]:
# Get all line indices from the MDBs
line_indices = [mdb.cLin for mdb in image_mdbs]

# Find unique lines
unique_lines = sorted(set(line_indices))

# Number of phase-encoding lines
n_line = len(unique_lines)

print(f"Unique phase-encoding line indices: {unique_lines}")
print(f"Total number of phase-encoding lines: {n_line}")

In [ ]:
from collections import Counter

# Collect shapes of all MDBs
shapes = [mdb.data.shape for mdb in image_mdbs]

# Count occurrences of each unique shape
shape_counts = Counter(shapes)

# Print initial shape
print(f"Initial MDB shape (channels, columns): {shapes[0]}")

# Print summary of all shapes
print("\nSummary of unique MDB shapes:")
for shape, count in shape_counts.items():
    print(f"  Shape {shape}: {count} MDB(s)")

# Optional: check if all shapes are consistent
if len(shape_counts) == 1:
    print("\nAll MDBs have the same shape. ")
else:
    print("\nSome MDBs have differing shapes. ")


In [ ]:
mdb = image_mdbs[0]
for attr in dir(mdb):
    if not attr.startswith("__"):
        print(attr)


In [ ]:
from collections import defaultdict

# Dictionary to collect unique values for each attribute
summary = defaultdict(set)

for mdb in image_mdbs:
    # Basic k-space info
    summary['echo'].add(getattr(mdb, 'cEco', None))
    summary['partition'].add(getattr(mdb, 'cPar', None))
    summary['line'].add(getattr(mdb, 'cLin', None))
    
    # Channels and columns
    n_channel, n_column = getattr(mdb, 'data', np.zeros((0,0))).shape
    summary['channels'].add(n_channel)
    summary['columns'].add(n_column)
    
    # Acquisition parameters (may not exist in all MDBs)
    summary['TR'].add(getattr(mdb, 'TR', None))
    summary['TE'].add(getattr(mdb, 'TE', None))
    summary['FA'].add(getattr(mdb, 'FA', None))
    
    # Slice / position info
    summary['slicePos'].add(getattr(mdb, 'slicePos', None))
    
    # Identifiers
    summary['scanId'].add(getattr(mdb, 'scanId', None))
    summary['scanCounter'].add(getattr(mdb, 'scanCounter', None))

# Print dataset summary
print("=== MDB Dataset Summary ===\n")
for key, values in summary.items():
    clean_values = sorted([v for v in values if v is not None])
    print(f"{key:12}: {len(clean_values)} unique -> {clean_values[:10]}{'...' if len(clean_values) > 10 else ''}")
    
# Optional: check consistency for channels and columns
if len(summary['channels']) == 1 and len(summary['columns']) == 1:
    print("\nAll MDBs have consistent channels and columns ")
else:
    print("\nSome MDBs have varying channels or columns ")


In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np

# # Loop over echoes, partitions, and channels
# for eco in range(kspace.shape[0]):       # echo dimension
#     for par in range(kspace.shape[1]):   # partition/slice dimension
#         for ch in range(kspace.shape[3]):  # channel dimension
            
#             # Pick k-space for this channel
#             ks = kspace[eco, par, :, ch, :]  # shape: [line, column]
            
#             # Reconstruct image using 2D IFFT
#             image = np.fft.ifft2(np.fft.ifftshift(ks))
#             image = np.abs(image)  # magnitude
            
#             # Plot k-space magnitude and reconstructed image
#             plt.figure(figsize=[12,5])
            
#             # k-space magnitude
#             plt.subplot(121)
#             plt.title(f'k-space magnitude (Echo {eco}, Partition {par}, Channel {ch})')
#             plt.imshow(np.abs(ks)**0.2, cmap='gray', origin='lower')
#             plt.axis('off')
            
#             # reconstructed image
#             plt.subplot(122)
#             plt.title(f'Reconstructed image (Echo {eco}, Partition {par}, Channel {ch})')
#             plt.imshow(image, cmap='gray', origin='lower')
#             plt.axis('off')
            
#             plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- helper functions ---
def ifftnd(data, axes=None):
    """N-dimensional inverse FFT with fftshift."""
    if axes is None:
        axes = tuple(range(data.ndim))
    return np.fft.ifftshift(np.fft.ifftn(np.fft.fftshift(data, axes=axes), axes=axes), axes=axes)

def rms_comb(image, coil_axis=0):
    """Combine coil images using root-sum-of-squares (RMS)."""
    return np.sqrt(np.sum(np.abs(image)**2, axis=coil_axis))

# --- STEP 1: filter all imaging mdbs ---
image_mdbs = [mdb for mdb in twix[-1]['mdb'] if mdb.is_image_scan()]

# --- STEP 2: determine k-space dimensions ---
n_echo = 1 + max([mdb.cEco for mdb in image_mdbs])
n_partition = 1 + max([mdb.cPar for mdb in image_mdbs])
n_line = 1 + max([mdb.cLin for mdb in image_mdbs])
n_channel, n_column = image_mdbs[0].data.shape

print(f"Expected k-space shape: echo {n_echo}, partition {n_partition}, line {n_line}, channel {n_channel}, column {n_column}")

# --- STEP 3: initialize empty k-space array ---
kspace = np.zeros([n_echo, n_partition, n_line, n_channel, n_column], dtype=np.complex64)

# --- STEP 4: fill k-space array ---
for mdb in image_mdbs:
    kspace[mdb.cEco, mdb.cPar, mdb.cLin] = mdb.data

# --- STEP 5: reconstruction and visualization ---
for eco in range(n_echo-3):
    for par in range(n_partition):
        ks = kspace[eco, par]  # shape: [line, channel, column]

        # --- reconstruct 2D image per partition ---
        # FFT along line and column axes
        image_per_coil = ifftnd(ks, axes=[0,2])
        # combine coils using RMS
        image_comb = rms_comb(image_per_coil, coil_axis=1)

        # --- visualization ---
        plt.figure(figsize=[12,5])

        # k-space magnitude (first coil)
        plt.subplot(121)
        plt.title(f'k-space magnitude (Echo {eco}, Partition {par})')
        #plt.imshow(np.log1p(np.abs(ks[:,0])), cmap='gray', origin='lower')
        plt.imshow(abs(ks[:,0])**0.2, cmap='gray', origin='lower')
        plt.axis('off')

        # reconstructed image
        plt.subplot(122)
        plt.title(f'Reconstructed image (Echo {eco}, Partition {par})')
        plt.imshow(np.abs(image_comb), cmap='gray', origin='lower')
        plt.axis('off')

        plt.show()


In [ ]:
# --- limit the number of plots ---
max_echoes_to_plot = min(2, n_echo)       # first 2 echoes
max_partitions_to_plot = min(3, n_partition)  # first 3 partitions
max_channels_to_plot = min(2, n_channel)    # first 2 channels

for eco in range(max_echoes_to_plot):       
    for par in range(max_partitions_to_plot):   
        for ch in range(max_channels_to_plot):  
            
            # Pick k-space for this echo/partition/channel
            ks = kspace[eco, par, :, ch, :]  # shape: [line, column]
            
            # Reconstruct image using 2D IFFT with proper centering
            image = ifftnd(ks, axes=[0,1])
            image = np.abs(image)  # magnitude
            
            # Plot k-space magnitude and reconstructed image
            plt.figure(figsize=[12,5])
            
            # k-space magnitude
            plt.subplot(121)
            plt.title(f'k-space magnitude (Echo {eco}, Partition {par}, Channel {ch})')
            plt.imshow(np.abs(ks)**0.2, cmap='gray', origin='lower')
            plt.axis('off')
            
            # Reconstructed image
            plt.subplot(122)
            plt.title(f'Reconstructed image (Echo {eco}, Partition {par}, Channel {ch})')
            plt.imshow(image, cmap='gray', origin='lower')
            plt.axis('off')
            
            plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- helper function ---
def ifftnd(data, axes=None):
    """N-dimensional inverse FFT with fftshift."""
    if axes is None:
        axes = tuple(range(data.ndim))
    return np.fft.ifftshift(np.fft.ifftn(np.fft.fftshift(data, axes=axes), axes=axes), axes=axes)

def rms_comb(image, coil_axis=0):
    """Combine coil/channel images using RMS."""
    return np.sqrt(np.sum(np.abs(image)**2, axis=coil_axis))

# --- limit the number of plots ---
max_echoes_to_plot = min(2, kspace.shape[0])        # first 2 echoes
max_partitions_to_plot = min(3, kspace.shape[1])    # first 3 partitions

for eco in range(max_echoes_to_plot):       
    for par in range(max_partitions_to_plot):   
        
        # Pick k-space for this echo/partition (all channels)
        ks = kspace[eco, par]  # shape: [lines, channels, columns]
        
        # Reconstruct images per channel using 2D IFFT
        image_per_channel = ifftnd(ks, axes=[0,2])  # FFT along line and column axes
        
        # Combine channels using RMS
        image_comb = rms_comb(image_per_channel, coil_axis=1)
        
        # Plot k-space magnitude of first channel and RMS image
        plt.figure(figsize=[12,5])
        
        # k-space magnitude (first channel)
        plt.subplot(121)
        plt.title(f'k-space magnitude (Echo {eco}, Partition {par}, first channel)')
        plt.imshow(np.abs(ks[:,0,:])**0.2, cmap='gray', origin='lower')
        plt.axis('off')
        
        # RMS reconstructed image
        plt.subplot(122)
        plt.title(f'Reconstructed image RMS (Echo {eco}, Partition {par})')
        plt.imshow(np.abs(image_comb), cmap='gray', origin='lower')
        plt.axis('off')
        
        plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- helper functions ---
def ifftnd(data, axes=None):
    """N-dimensional inverse FFT with fftshift."""
    if axes is None:
        axes = tuple(range(data.ndim))
    return np.fft.ifftshift(np.fft.ifftn(np.fft.fftshift(data, axes=axes), axes=axes), axes=axes)

def rms_comb(image, coil_axis=0):
    """Combine coil/channel images using RMS."""
    return np.sqrt(np.sum(np.abs(image)**2, axis=coil_axis))

# --- choose which echo to visualize ---
echo_index = 0  # first echo
n_partitions = kspace.shape[1]

# --- reconstruct 3D volume by combining all channels ---
volume = []
for par in range(n_partitions):
    ks = kspace[echo_index, par]  # shape: [lines, channels, columns]
    image_per_channel = ifftnd(ks, axes=[0,2])
    image_rms = rms_comb(image_per_channel, coil_axis=1)  # combine channels
    volume.append(image_rms)

volume = np.stack(volume, axis=0)  # shape: [partition, height, width]

# --- plot all slices in a grid ---
cols = 5  # number of slices per row
rows = int(np.ceil(n_partitions / cols))

fig, axes = plt.subplots(rows, cols, figsize=(15, 3*rows))
axes = axes.flatten()

for i in range(len(axes)):
    if i < n_partitions:
        axes[i].imshow(volume[i], cmap='gray', origin='lower')
        axes[i].set_title(f'Partition {i}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# -----------------------------
# Helper functions
# -----------------------------
def ifftnd(data, axes=None):
    """N-dimensional inverse FFT with fftshift."""
    if axes is None:
        axes = tuple(range(data.ndim))
    return np.fft.ifftshift(np.fft.ifftn(np.fft.fftshift(data, axes=axes), axes=axes), axes=axes)

def rms_comb(image, coil_axis=0):
    """Combine coil/channel images using RMS."""
    return np.sqrt(np.sum(np.abs(image)**2, axis=coil_axis))

def reconstruct_rms_volume(kspace_subset):
    """
    Reconstruct 3D RMS images for all echoes.
    kspace_subset: [echo, partition, line, channel, column]
    Returns: volume [echo, partition, height, width]
    """
    n_echo, n_part, _, _, _ = kspace_subset.shape
    volume = []
    
    for eco in range(n_echo):
        vol_echo = []
        for par in range(n_part):
            ks = kspace_subset[eco, par]  # [line, channel, column]
            img_per_channel = ifftnd(ks, axes=[0,2])  # IFFT along line & column
            img_rms = rms_comb(img_per_channel, coil_axis=1)  # combine channels
            vol_echo.append(np.abs(img_rms))
        vol_echo = np.stack(vol_echo, axis=0)  # [partition, height, width]
        volume.append(vol_echo)
    
    return np.stack(volume, axis=0)  # [echo, partition, height, width]

# -----------------------------
# Step 1: Normalize k-space
# -----------------------------
kspace_norm = kspace / np.max(np.abs(kspace))  # scale to [-1,1] roughly

# -----------------------------
# Step 2: Split partitions into train/val/test
# -----------------------------
n_partition = kspace_norm.shape[1]
partition_indices = np.arange(n_partition)

train_idx, temp_idx = train_test_split(partition_indices, test_size=0.3, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

kspace_train = kspace_norm[:, train_idx, :, :, :]  # all echoes
kspace_val   = kspace_norm[:, val_idx, :, :, :]
kspace_test  = kspace_norm[:, test_idx, :, :, :]

# -----------------------------
# Step 3: Reconstruct RMS 3D volumes for all echoes
# -----------------------------
images_train = reconstruct_rms_volume(kspace_train)
images_val   = reconstruct_rms_volume(kspace_val)
images_test  = reconstruct_rms_volume(kspace_test)

print("Train volume shape:", images_train.shape)
print("Val volume shape:", images_val.shape)
print("Test volume shape:", images_test.shape)

# -----------------------------
# Step 4: Optional visualization of slices for one echo
# -----------------------------
def visualize_volume_slices(volume, echo_idx=0, cols=5):
    vol_echo = volume[echo_idx]  # [partition, height, width]
    n_part = vol_echo.shape[0]
    rows = int(np.ceil(n_part / cols))
    
    fig, axes = plt.subplots(rows, cols, figsize=(15, 3*rows))
    axes = axes.flatten()
    
    for i in range(len(axes)):
        if i < n_part:
            axes[i].imshow(vol_echo[i], cmap='gray', origin='lower')
            axes[i].set_title(f'Echo {echo_idx}, Slice {i}')
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

# Example: visualize first echo of train set
visualize_volume_slices(images_train, echo_idx=0)


In [ ]:
print(kspace.shape)

In [ ]:
##Unet good
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim_metric, peak_signal_noise_ratio as psnr_metric
import torchvision.transforms.functional as TF

# ===============================================================
# === 1. Helper Functions =======================================
# ===============================================================

def ifftnd(data, axes=None):
    """N-dimensional inverse FFT with fftshift."""
    if axes is None:
        axes = tuple(range(data.ndim))
    return np.fft.ifftshift(np.fft.ifftn(np.fft.fftshift(data, axes=axes), axes=axes), axes=axes)

def rms_comb(image, coil_axis=0):
    """Combine coil images using root-sum-of-squares (RMS)."""
    return np.sqrt(np.sum(np.abs(image)**2, axis=coil_axis))

def undersample_kspace(kspace_slice, accel_factor=4):
    """Uniform undersampling in phase-encode direction."""
    n_lines = kspace_slice.shape[0]
    mask = np.zeros(n_lines, dtype=bool)
    mask[::accel_factor] = True
    mask[n_lines//2 - 3 : n_lines//2 + 3] = True  # preserve center
    kspace_under = np.zeros_like(kspace_slice)
    kspace_under[mask] = kspace_slice[mask]
    return kspace_under, mask

# ===============================================================
# === 2. K-space Loading from .twix =============================
# ===============================================================

# --- filter all imaging mdbs ---
image_mdbs = [mdb for mdb in twix[-1]['mdb'] if mdb.is_image_scan()]

# --- determine k-space dimensions ---
n_echo = 1 + max([mdb.cEco for mdb in image_mdbs])
n_partition = 1 + max([mdb.cPar for mdb in image_mdbs])
n_line = 1 + max([mdb.cLin for mdb in image_mdbs])
n_channel, n_column = image_mdbs[0].data.shape

print(f"K-space shape: echo={n_echo}, partition={n_partition}, line={n_line}, channel={n_channel}, column={n_column}")

# --- initialize and fill k-space array ---
kspace = np.zeros([n_echo, n_partition, n_line, n_channel, n_column], dtype=np.complex64)
for mdb in image_mdbs:
    kspace[mdb.cEco, mdb.cPar, mdb.cLin] = mdb.data

# ===============================================================
# === 3. PyTorch Dataset (all echoes + partitions) =============
# ===============================================================

class MRIDataset(Dataset):
    def __init__(self, kspace, accel_factor=4, augment=False):
        self.kspace = kspace
        self.accel_factor = accel_factor
        self.augment = augment
        # Create all (echo, partition) pairs
        self.echo_partition_pairs = [
            (eco, par)
            for eco in range(kspace.shape[0])
            for par in range(kspace.shape[1])
        ]

    def __len__(self):
        return len(self.echo_partition_pairs)

    def __getitem__(self, idx):
        eco, par = self.echo_partition_pairs[idx]
        ks_full = self.kspace[eco, par]  # [line, coil, column]

        kspace_under, mask = undersample_kspace(ks_full, accel_factor=self.accel_factor)

        # Reconstruct
        img_full = ifftnd(ks_full, axes=[0,2])
        img_under = ifftnd(kspace_under, axes=[0,2])

        # Combine coils
        img_full = rms_comb(img_full, coil_axis=1)
        img_under = rms_comb(img_under, coil_axis=1)

        # Normalize
        img_full = img_full / np.max(img_full)
        img_under = img_under / np.max(img_under)

        # Convert to tensors
        img_full = torch.tensor(img_full, dtype=torch.float32).unsqueeze(0)
        img_under = torch.tensor(img_under, dtype=torch.float32).unsqueeze(0)

        # Optional augmentations
        if self.augment:
            if np.random.rand() < 0.5:
                img_under = TF.hflip(img_under)
                img_full = TF.hflip(img_full)
            if np.random.rand() < 0.5:
                img_under = TF.vflip(img_under)
                img_full = TF.vflip(img_full)

        return {'image_under': img_under, 'image_gt': img_full}

# --- Create dataset ---
dataset = MRIDataset(kspace, accel_factor=4, augment=True)

# --- Split into train/val/test ---
n_total = len(dataset)
train_size = int(0.7 * n_total)
val_size = int(0.15 * n_total)
test_size = n_total - train_size - val_size
train_data, val_data, test_data = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])

print(f"Train={len(train_data)}, Val={len(val_data)}, Test={len(test_data)}")

# --- DataLoaders ---
train_loader = DataLoader(train_data, batch_size=8, shuffle=True)
val_loader = DataLoader(val_data, batch_size=8)
test_loader = DataLoader(test_data, batch_size=8)

# ===============================================================
# === 4. U-Net Model ============================================
# ===============================================================

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1):
        super().__init__()
        self.down1 = DoubleConv(in_ch, 64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)
        self.down4 = DoubleConv(256, 512)
        self.pool = nn.MaxPool2d(2)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv3 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv2 = DoubleConv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv1 = DoubleConv(128, 64)
        self.outc = nn.Conv2d(64, out_ch, 1)
    def forward(self, x):
        c1 = self.down1(x)
        p1 = self.pool(c1)
        c2 = self.down2(p1)
        p2 = self.pool(c2)
        c3 = self.down3(p2)
        p3 = self.pool(c3)
        c4 = self.down4(p3)
        u3 = self.up3(c4)
        c3 = self.conv3(torch.cat([u3, c3], dim=1))
        u2 = self.up2(c3)
        c2 = self.conv2(torch.cat([u2, c2], dim=1))
        u1 = self.up1(c2)
        c1 = self.conv1(torch.cat([u1, c1], dim=1))
        out = self.outc(c1)
        return out

# ===============================================================
# === 5. Training ===============================================
# ===============================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.MSELoss()

epochs = 500
train_loss_history, val_loss_history = [], []

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        img_under = batch['image_under'].to(device)
        img_gt = batch['image_gt'].to(device)
        pred = model(img_under)
        loss = criterion(pred, img_gt)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_train = total_loss / len(train_loader)

    # Validation
    model.eval()
    with torch.no_grad():
        val_loss = 0
        for batch in val_loader:
            img_under = batch['image_under'].to(device)
            img_gt = batch['image_gt'].to(device)
            pred = model(img_under)
            loss = criterion(pred, img_gt)
            val_loss += loss.item()
        avg_val = val_loss / len(val_loader)

    train_loss_history.append(avg_train)
    val_loss_history.append(avg_val)
    if epoch % 100==0:
        
        print(f"Epoch {epoch+1}/{epochs} - Train: {avg_train:.4f} - Val: {avg_val:.4f}")

# ===============================================================
# === 6. Evaluation (PSNR + SSIM) ===============================
# ===============================================================

model.eval()
psnr_scores, ssim_scores = [], []

with torch.no_grad():
    for batch in test_loader:
        img_under = batch['image_under'].to(device)
        img_gt = batch['image_gt'].to(device)
        pred = model(img_under)

        for i in range(pred.size(0)):
            pred_np = pred[i,0].cpu().numpy()
            gt_np = img_gt[i,0].cpu().numpy()
            psnr_scores.append(psnr_metric(gt_np, pred_np, data_range=1.0))
            ssim_scores.append(ssim_metric(gt_np, pred_np, data_range=1.0))

print(f"Mean PSNR: {np.mean(psnr_scores):.3f}, Mean SSIM: {np.mean(ssim_scores):.3f}")

# ===============================================================
# === 7. Visualization ==========================================
# ===============================================================

plt.figure()
plt.plot(train_loss_history, label='Train')
plt.plot(val_loss_history, label='Val')
plt.legend(); plt.title("Training Loss"); plt.xlabel("Epoch"); plt.ylabel("MSE Loss"); plt.show()

# Show example reconstruction
batch = next(iter(test_loader))
img_under = batch['image_under'].to(device)
img_gt = batch['image_gt'].to(device)
with torch.no_grad():
    pred = model(img_under)

plt.figure(figsize=(12,4))
plt.subplot(1,3,1); plt.imshow(img_under[0,0].cpu(), cmap='gray'); plt.title("Undersampled")
plt.subplot(1,3,2); plt.imshow(pred[0,0].cpu(), cmap='gray'); plt.title("Reconstructed (U-Net)")
plt.subplot(1,3,3); plt.imshow(img_gt[0,0].cpu(), cmap='gray'); plt.title("Ground Truth")
plt.show()


In [ ]:
model.eval()
psnr_scores, ssim_scores = [], []

all_reconstructions = []  # optional: store reconstructed images
all_gt = []
all_under = []

with torch.no_grad():
    for batch in test_loader:
        img_under = batch['image_under'].to(device)
        img_gt = batch['image_gt'].to(device)
        pred = model(img_under)

        for i in range(pred.size(0)):
            pred_np = pred[i,0].cpu().numpy()
            gt_np = img_gt[i,0].cpu().numpy()
            under_np = img_under[i,0].cpu().numpy()

            psnr_scores.append(psnr_metric(gt_np, pred_np, data_range=1.0))
            ssim_scores.append(ssim_metric(gt_np, pred_np, data_range=1.0))

            all_reconstructions.append(pred_np)
            all_gt.append(gt_np)
            all_under.append(under_np)

# Print final metrics
print(f"Test Set: {len(all_reconstructions)} samples")
print(f"Mean PSNR: {np.mean(psnr_scores):.3f}, Mean SSIM: {np.mean(ssim_scores):.3f}")

# Optional: visualize a few random test samples
n_show = 23
for idx in np.random.choice(len(all_reconstructions), n_show, replace=False):
    plt.figure(figsize=(12,4))
    plt.subplot(1,3,1); plt.imshow(all_under[idx], cmap='gray'); plt.title("Undersampled")
    plt.subplot(1,3,2); plt.imshow(all_reconstructions[idx], cmap='gray'); plt.title("Reconstructed")
    plt.subplot(1,3,3); plt.imshow(all_gt[idx], cmap='gray'); plt.title("Ground Truth")
    plt.show()

In [ ]:
# ===============================================================
# === ResUNet MRI Reconstruction using Pretrained ResNet =======
# ===============================================================

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim_metric, peak_signal_noise_ratio as psnr_metric
import torch.nn.functional as F
import torchvision.models as models

# ===============================================================
# === 1. Helper Functions =======================================
# ===============================================================

def ifftnd(data, axes=None):
    if axes is None:
        axes = tuple(range(data.ndim))
    return np.fft.ifftshift(np.fft.ifftn(np.fft.fftshift(data, axes=axes), axes=axes), axes=axes)

def rms_comb(image, coil_axis=0):
    return np.sqrt(np.sum(np.abs(image)**2, axis=coil_axis))

def undersample_kspace(kspace_slice, accel_factor=4):
    n_lines = kspace_slice.shape[0]
    mask = np.zeros(n_lines, dtype=bool)
    mask[::accel_factor] = True
    mask[n_lines//2 - 3 : n_lines//2 + 3] = True  # keep center
    kspace_under = np.zeros_like(kspace_slice)
    kspace_under[mask] = kspace_slice[mask]
    return kspace_under, mask

# ===============================================================
# === 2. MRIDataset ============================================
# ===============================================================

class MRIDataset(Dataset):
    def __init__(self, kspace, accel_factor=4, augment=False):
        self.kspace = kspace
        self.accel_factor = accel_factor
        self.augment = augment
        self.echo_partition_pairs = [
            (eco, par)
            for eco in range(kspace.shape[0])
            for par in range(kspace.shape[1])
        ]

    def __len__(self):
        return len(self.echo_partition_pairs)

    def __getitem__(self, idx):
        eco, par = self.echo_partition_pairs[idx]
        ks_full = self.kspace[eco, par]  # [line, coil, column]
        kspace_under, mask = undersample_kspace(ks_full, accel_factor=self.accel_factor)

        # Reconstruct images
        img_full = ifftnd(ks_full, axes=[0,2])
        img_under = ifftnd(kspace_under, axes=[0,2])

        img_full = rms_comb(img_full, coil_axis=1)
        img_under = rms_comb(img_under, coil_axis=1)

        # Normalize
        img_full = img_full / np.max(img_full)
        img_under = img_under / np.max(img_under)

        # Convert to torch tensors
        img_full = torch.tensor(img_full, dtype=torch.float32).unsqueeze(0)
        img_under = torch.tensor(img_under, dtype=torch.float32).unsqueeze(0)

        if self.augment:
            if np.random.rand() < 0.5:
                img_under = torch.flip(img_under, dims=[2])
                img_full = torch.flip(img_full, dims=[2])
            if np.random.rand() < 0.5:
                img_under = torch.flip(img_under, dims=[1])
                img_full = torch.flip(img_full, dims=[1])

        return {'image_under': img_under, 'image_gt': img_full}

# ===============================================================
# === 3. ResUNet with Pretrained ResNet Encoder =================
# ===============================================================

class ResUNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
        # Modify first conv layer to accept 1-channel input
        resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        # Encoder without avgpool + fc
        self.encoder = nn.Sequential(*list(resnet.children())[:-2])

        # Decoder to reconstruct image
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, kernel_size=1)
        )

    def forward(self, x):
        x_enc = self.encoder(x)
        x_dec = self.decoder(x_enc)
        # Upsample to original size
        x_dec = F.interpolate(x_dec, size=(x.shape[2], x.shape[3]), mode='bilinear', align_corners=False)
        return x_dec

# ===============================================================
# === 4. Load k-space and Create Dataset =======================
# ===============================================================

# Example: replace with your actual loaded k-space array
# kspace shape: [echo, partition, line, coil, column]
# kspace = np.load("your_kspace.npy")

dataset = MRIDataset(kspace, accel_factor=4, augment=True)
n_total = len(dataset)
train_size = int(0.7 * n_total)
val_size = int(0.15 * n_total)
test_size = n_total - train_size - val_size
train_data, val_data, test_data = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_data, batch_size=4, shuffle=True)
val_loader = DataLoader(val_data, batch_size=4)
test_loader = DataLoader(test_data, batch_size=4)

# ===============================================================
# === 5. Model, Loss, Optimizer ================================
# ===============================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ResUNet(pretrained=True).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.MSELoss()

# ===============================================================
# === 6. Training ==============================================
# ===============================================================

epochs = 200
train_loss_history, val_loss_history = [], []

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        img_under = batch['image_under'].to(device)
        img_gt = batch['image_gt'].to(device)
        pred = model(img_under)
        loss = criterion(pred, img_gt)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_train = total_loss / len(train_loader)

    # Validation
    model.eval()
    with torch.no_grad():
        val_loss = 0
        for batch in val_loader:
            img_under = batch['image_under'].to(device)
            img_gt = batch['image_gt'].to(device)
            pred = model(img_under)
            loss = criterion(pred, img_gt)
            val_loss += loss.item()
        avg_val = val_loss / len(val_loader)

    train_loss_history.append(avg_train)
    val_loss_history.append(avg_val)

    if epoch % 20 == 0:
        print(f"Epoch {epoch+1}/{epochs} - Train: {avg_train:.6f} - Val: {avg_val:.6f}")

# ===============================================================
# === 7. Evaluation (PSNR + SSIM) =============================
# ===============================================================

model.eval()
psnr_scores, ssim_scores = [], []

with torch.no_grad():
    for batch in test_loader:
        img_under = batch['image_under'].to(device)
        img_gt = batch['image_gt'].to(device)
        pred = model(img_under)

        for i in range(pred.size(0)):
            pred_np = pred[i,0].cpu().numpy()
            gt_np = img_gt[i,0].cpu().numpy()
            psnr_scores.append(psnr_metric(gt_np, pred_np, data_range=1.0))
            ssim_scores.append(ssim_metric(gt_np, pred_np, data_range=1.0))

print(f"Test PSNR: {np.mean(psnr_scores):.3f}, Test SSIM: {np.mean(ssim_scores):.3f}")

# ===============================================================
# === 8. Visualization =========================================
# ===============================================================

# Loss curves
plt.figure()
plt.plot(train_loss_history, label='Train')
plt.plot(val_loss_history, label='Val')
plt.legend(); plt.title("Training Loss"); plt.xlabel("Epoch"); plt.ylabel("MSE Loss"); plt.show()

# Example reconstructions
batch = next(iter(test_loader))
img_under = batch['image_under'].to(device)
img_gt = batch['image_gt'].to(device)
with torch.no_grad():
    pred = model(img_under)

plt.figure(figsize=(12,4))
plt.subplot(1,3,1); plt.imshow(img_under[0,0].cpu(), cmap='gray'); plt.title("Undersampled")
plt.subplot(1,3,2); plt.imshow(pred[0,0].cpu(), cmap='gray'); plt.title("Reconstructed (ResUNet)")
plt.subplot(1,3,3); plt.imshow(img_gt[0,0].cpu(), cmap='gray'); plt.title("Ground Truth")
plt.show()


In [ ]:
# ===============================================================
# === Attention U-Net MRI Reconstruction =======================
# ===============================================================

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim_metric, peak_signal_noise_ratio as psnr_metric
import torch.nn.functional as F

# ===============================================================
# === 1. Helper Functions =======================================
# ===============================================================

def ifftnd(data, axes=None):
    if axes is None:
        axes = tuple(range(data.ndim))
    return np.fft.ifftshift(np.fft.ifftn(np.fft.fftshift(data, axes=axes), axes=axes), axes=axes)

def rms_comb(image, coil_axis=0):
    return np.sqrt(np.sum(np.abs(image)**2, axis=coil_axis))

def undersample_kspace(kspace_slice, accel_factor=4):
    n_lines = kspace_slice.shape[0]
    mask = np.zeros(n_lines, dtype=bool)
    mask[::accel_factor] = True
    mask[n_lines//2 - 3 : n_lines//2 + 3] = True  # keep center
    kspace_under = np.zeros_like(kspace_slice)
    kspace_under[mask] = kspace_slice[mask]
    return kspace_under, mask

# ===============================================================
# === 2. MRIDataset ============================================
# ===============================================================

class MRIDataset(Dataset):
    def __init__(self, kspace, accel_factor=4, augment=False):
        self.kspace = kspace
        self.accel_factor = accel_factor
        self.augment = augment
        self.echo_partition_pairs = [
            (eco, par)
            for eco in range(kspace.shape[0])
            for par in range(kspace.shape[1])
        ]

    def __len__(self):
        return len(self.echo_partition_pairs)

    def __getitem__(self, idx):
        eco, par = self.echo_partition_pairs[idx]
        ks_full = self.kspace[eco, par]  # [line, coil, column]
        kspace_under, mask = undersample_kspace(ks_full, accel_factor=self.accel_factor)

        # Reconstruct images
        img_full = ifftnd(ks_full, axes=[0,2])
        img_under = ifftnd(kspace_under, axes=[0,2])

        img_full = rms_comb(img_full, coil_axis=1)
        img_under = rms_comb(img_under, coil_axis=1)

        # Normalize
        img_full = img_full / np.max(img_full)
        img_under = img_under / np.max(img_under)

        # Convert to torch tensors
        img_full = torch.tensor(img_full, dtype=torch.float32).unsqueeze(0)
        img_under = torch.tensor(img_under, dtype=torch.float32).unsqueeze(0)

        if self.augment:
            if np.random.rand() < 0.5:
                img_under = torch.flip(img_under, dims=[2])
                img_full = torch.flip(img_full, dims=[2])
            if np.random.rand() < 0.5:
                img_under = torch.flip(img_under, dims=[1])
                img_full = torch.flip(img_full, dims=[1])

        return {'image_under': img_under, 'image_gt': img_full}

# ===============================================================
# === 3. Attention Block =======================================
# ===============================================================

class AttentionBlock(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

# ===============================================================
# === 4. Double Conv ===========================================
# ===============================================================

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

# ===============================================================
# === 5. Attention U-Net =======================================
# ===============================================================

class AttentionUNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1):
        super().__init__()
        self.down1 = DoubleConv(in_ch, 64)
        self.pool = nn.MaxPool2d(2)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)
        self.down4 = DoubleConv(256, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.att3 = AttentionBlock(F_g=256, F_l=256, F_int=128)
        self.conv3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.att2 = AttentionBlock(F_g=128, F_l=128, F_int=64)
        self.conv2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.att1 = AttentionBlock(F_g=64, F_l=64, F_int=32)
        self.conv1 = DoubleConv(128, 64)

        self.outc = nn.Conv2d(64, out_ch, 1)

    def forward(self, x):
        c1 = self.down1(x)
        p1 = self.pool(c1)
        c2 = self.down2(p1)
        p2 = self.pool(c2)
        c3 = self.down3(p2)
        p3 = self.pool(c3)
        c4 = self.down4(p3)

        u3 = self.up3(c4)
        c3 = self.att3(u3, c3)
        c3 = self.conv3(torch.cat([u3, c3], dim=1))

        u2 = self.up2(c3)
        c2 = self.att2(u2, c2)
        c2 = self.conv2(torch.cat([u2, c2], dim=1))

        u1 = self.up1(c2)
        c1 = self.att1(u1, c1)
        c1 = self.conv1(torch.cat([u1, c1], dim=1))

        out = self.outc(c1)
        # Upsample to input size
        out = F.interpolate(out, size=(x.shape[2], x.shape[3]), mode='bilinear', align_corners=False)
        return out

# ===============================================================
# === 6. Load Dataset ==========================================
# ===============================================================

# Example: replace with your actual loaded k-space array
# kspace shape: [6, 26, 192, 30, 384]
# kspace = np.load("your_kspace.npy")

dataset = MRIDataset(kspace, accel_factor=4, augment=True)
n_total = len(dataset)
train_size = int(0.7 * n_total)
val_size = int(0.15 * n_total)
test_size = n_total - train_size - val_size
train_data, val_data, test_data = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_data, batch_size=4, shuffle=True)
val_loader = DataLoader(val_data, batch_size=4)
test_loader = DataLoader(test_data, batch_size=4)

# ===============================================================
# === 7. Model, Loss, Optimizer =================================
# ===============================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AttentionUNet(in_ch=1, out_ch=1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.MSELoss()

# ===============================================================
# === 8. Training ===============================================
# ===============================================================

epochs = 200
train_loss_history, val_loss_history = [], []

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        img_under = batch['image_under'].to(device)
        img_gt = batch['image_gt'].to(device)
        pred = model(img_under)
        loss = criterion(pred, img_gt)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_train = total_loss / len(train_loader)

    model.eval()
    with torch.no_grad():
        val_loss = 0
        for batch in val_loader:
            img_under = batch['image_under'].to(device)
            img_gt = batch['image_gt'].to(device)
            pred = model(img_under)
            loss = criterion(pred, img_gt)
            val_loss += loss.item()
        avg_val = val_loss / len(val_loader)

    train_loss_history.append(avg_train)
    val_loss_history.append(avg_val)

    if epoch % 20 == 0:
        print(f"Epoch {epoch+1}/{epochs} - Train: {avg_train:.6f} - Val: {avg_val:.6f}")

# ===============================================================
# === 9. Evaluation (PSNR + SSIM) =============================
# ===============================================================

model.eval()
psnr_scores, ssim_scores = [], []

with torch.no_grad():
    for batch in test_loader:
        img_under = batch['image_under'].to(device)
        img_gt = batch['image_gt'].to(device)
        pred = model(img_under)

        for i in range(pred.size(0)):
            pred_np = pred[i,0].cpu().numpy()
            gt_np = img_gt[i,0].cpu().numpy()
            psnr_scores.append(psnr_metric(gt_np, pred_np, data_range=1.0))
            ssim_scores.append(ssim_metric(gt_np, pred_np, data_range=1.0))

print(f"Test PSNR: {np.mean(psnr_scores):.3f}, Test SSIM: {np.mean(ssim_scores):.3f}")

# ===============================================================
# === 10. Visualization ========================================
# ===============================================================

# Loss curves
plt.figure()
plt.plot(train_loss_history, label='Train')
plt.plot(val_loss_history, label='Val')
plt.legend(); plt.title("Training Loss"); plt.xlabel("Epoch"); plt.ylabel("MSE Loss"); plt.show()

# Example reconstruction
batch = next(iter(test_loader))
img_under = batch['image_under'].to(device)
img_gt = batch['image_gt'].to(device)
with torch.no_grad():
    pred = model(img_under)

plt.figure(figsize=(12,4))
plt.subplot(1,3,1); plt.imshow(img_under[0,0].cpu(), cmap='gray'); plt.title("Undersampled")
plt.subplot(1,3,2); plt.imshow(pred[0,0].cpu(), cmap='gray');
